# Experiment V1: Faster R-CNN Baseline for Object Detection

**Student ID:** 25509225  
**Experiment:** V1 - Faster R-CNN Baseline (Standard Model)  
**Model:** Faster R-CNN with ResNet50+FPN (no customization)

---

## Overview

This experiment trains a standard Faster R-CNN model on the solar panel damage detection dataset without any architectural modifications. This serves as the baseline for comparison with customized models (V2: Deeper Backbone, V3: Shallower Backbone).

## Cell 1: Load Modules

In [ ]:
# Load shared modules from FasterRCNN_modules.ipynb
%run ./FasterRCNN_modules.ipynb

# Load data loading utilities
%run ./FasterRCNN_DataLoader.ipynb

print("✓ All modules loaded successfully")

## Cell 2: Configuration

In [ ]:
# ============================================================================
# Experiment V1: Faster R-CNN Baseline Configuration
# ============================================================================

# === Dataset Configuration ===
STUDENT_ID = "25509225"
# Use absolute path for SageMaker environment
DATA_ROOT = f"/home/sagemaker-user/CNN_A2/data/{STUDENT_ID}/Object_Detection/coco"
ANNOTATION_FORMAT = 'coco'
CLASS_NAMES = ['Cell', 'Cell-Multi', 'No-Anomaly', 'Shadowing', 'Unclassified']

# =========================================================
# Version 1 — Current Baseline (Reference)
# Goal:
# Current reference configuration
# =========================================================

MODEL_CONFIG_V1 = {
    'num_classes': len(CLASS_NAMES) + 1,
    'pretrained': True,
    'min_size': 640,
    'max_size': 640,
    'customize_type': None,

    'anchor_sizes': ((4,), (8,), (16,), (32,), (64,)),
    'anchor_aspect_ratios': ((0.5, 1.0, 2.0),) * 5,

    'rpn_pre_nms_top_n_train': 4000,
    'rpn_post_nms_top_n_train': 2000,
    'rpn_pre_nms_top_n_test': 2000,
    'rpn_post_nms_top_n_test': 1000,

    'rpn_nms_thresh': 0.85,
    'box_nms_thresh': 0.6
}

TRAINING_CONFIG_V1 = {
    'learning_rate': 0.005,
    'batch_size': 4,
    'epochs': 120,
    'optimizer': 'sgd',
    'momentum': 0.9,
    'weight_decay': 5e-4,
    'patience': 15,
}

# === Output Directory ===
output_dir = Path('/home/sagemaker-user/CNN_A2/notebooks/detection_FasterRCNN/outputs/detection_fasterrcnn_baseline_v1')
output_dir.mkdir(parents=True, exist_ok=True)

# Print experiment info
print("=" * 80)
print("EXPERIMENT V1: Faster R-CNN Baseline")
print("=" * 80)
print(f"\nDataset Configuration:")
print(f"  Student ID: {STUDENT_ID}")
print(f"  Data Root: {DATA_ROOT}")
print(f"  Annotation Format: {ANNOTATION_FORMAT}")
print(f"  Classes: {CLASS_NAMES}")
print(f"  Number of Classes: {len(CLASS_NAMES)} (+ background)")
print(f"\nModel Configuration:")
print(f"  Backbone: ResNet50+FPN (Standard)")
print(f"  Input Size: {MODEL_CONFIG_V1['min_size']}x{MODEL_CONFIG_V1['max_size']}")
print(f"  Pretrained: {MODEL_CONFIG_V1['pretrained']}")
print(f"  Customization: None (Baseline)")
print(f"\nTraining Configuration:")
print(f"  Learning Rate: {TRAINING_CONFIG_V1['learning_rate']}")
print(f"  Batch Size: {TRAINING_CONFIG_V1['batch_size']}")
print(f"  Epochs: {TRAINING_CONFIG_V1['epochs']}")
print(f"  Optimizer: {TRAINING_CONFIG_V1['optimizer']}")
print(f"  Weight Decay: {TRAINING_CONFIG_V1['weight_decay']}")
print(f"  Early Stopping Patience: {TRAINING_CONFIG_V1['patience']} epochs")
print(f"\nOutput Directory: {output_dir}")

## Cell 3: Step 1 - Create DataLoaders

In [ ]:
# ============================================================================
# Step 1: Create DataLoaders
# ============================================================================

print("\n[1/5] Creating dataloaders...")
print("=" * 80)

# create_faster_rcnn_dataloaders is already imported from FasterRCNN_DataLoader.ipynb

train_loader, val_loader, test_loader = create_faster_rcnn_dataloaders(
    data_root=DATA_ROOT,
    batch_size=TRAINING_CONFIG_V1['batch_size'],
    num_workers=4,
    annotation_format=ANNOTATION_FORMAT,
    class_names=CLASS_NAMES
)

print(f'\nDataset loaded successfully:')
print(f'  Train samples: {len(train_loader.dataset)}')
print(f'  Val samples: {len(val_loader.dataset)}')
print(f'  Test samples: {len(test_loader.dataset)}')
print(f'  Invalid bboxes filtered:')
print(f'    Train: {train_loader.dataset.skipped_annotations}')
print(f'    Val: {val_loader.dataset.skipped_annotations}')
print(f'    Test: {test_loader.dataset.skipped_annotations}')
print("\n✓ Dataloaders created successfully")

## Cell 4: Step 2 - Initialize Model

In [ ]:
# ============================================================================
# Step 2: Initialize Faster R-CNN Model
# ============================================================================

print("\n[2/5] Initializing Faster R-CNN model...")
print("=" * 80)

# Create model
model = FasterRCNNDetector(**MODEL_CONFIG_V1)

print(f'Number of classes: {MODEL_CONFIG_V1["num_classes"]} ({len(CLASS_NAMES)} + background)')
print(f'Image size: {MODEL_CONFIG_V1["min_size"]}x{MODEL_CONFIG_V1["max_size"]}')
print(f'Pretrained backbone: {MODEL_CONFIG_V1["pretrained"]}')
print(f'Customization: None (Baseline)')

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}, Trainable: {trainable_params:,}')
print(f'Approximate: ~{total_params/1e6:.1f}M parameters')
print("\n✓ Model initialized successfully")

## Cell 5: Step 3 - Train Model

In [ ]:
# ============================================================================
# Step 3: Train Model
# ============================================================================

print("\n[3/5] Training model...")
print("=" * 80)

# Initialize trainer
trainer = FasterRCNNTrainer(TRAINING_CONFIG_V1)

# Train model
history = trainer.train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=str(output_dir / 'training')
)

print(f'\nTraining completed!')
print(f'Best training loss: {history["best_loss"]:.4f}')
print(f'Total epochs: {history["total_epochs"]}')
print(f'Early stopped: {history["early_stopped"]}')
print("\n✓ Model training completed successfully")

## Cell 6: Step 4 - Evaluate on Test Set

In [ ]:
# ============================================================================
# Step 4: Evaluate on Test Set
# ============================================================================

print("\n[4/5] Evaluating model on test set...")
print("=" * 80)

# Initialize evaluator
evaluator = DetectionEvaluator()

# Evaluate model
metrics = evaluator.evaluate_fasterrcnn(
    model=model,
    test_loader=test_loader,
    output_dir=str(output_dir / 'evaluation'),
    class_names=CLASS_NAMES
)

print(f'\nTest Set Metrics:')
print(f'  mAP@0.5: {metrics["mAP50"]:.4f}')
print(f'  mAP@0.5:0.95: {metrics["mAP50_95"]:.4f}')
print(f'  Precision: {metrics["precision"]:.4f}')
print(f'  Recall: {metrics["recall"]:.4f}')
print("\n✓ Model evaluation completed successfully")

## Cell 7: Visualization

In [ ]:
# ============================================================================
# Visualization 1: Detection Results (Bounding Boxes)
# ============================================================================

print("\n=== Detection Results Visualization ===")
print("=" * 80)

# Display detection results inline
fig = evaluator.plot_detection_results(model, test_loader, num_samples=5)
if fig is not None:
    plt.show()
    print("\n✓ Detection results displayed")
else:
    print("Warning: Could not generate detection visualization")

In [ ]:
# ============================================================================
# Visualization 2: Training Loss Curves
# ============================================================================

print("\n=== Training Loss Curves ===")
print("=" * 80)

# Display saved loss curve image
fig = evaluator.plot_training_losses(str(output_dir))
if fig is not None:
    plt.show()
    print("\n✓ Training loss curves displayed")
else:
    print("Warning: Could not display training loss curves")
    print("This may be because loss_curve.png was not generated.")

In [ ]:
# ============================================================================
# Visualization 3: Validation mAP Curves
# ============================================================================

print("\n=== Validation mAP Curves ===")
print("=" * 80)

# Display saved mAP curve image
fig = evaluator.plot_validation_mAP(str(output_dir))
if fig is not None:
    plt.show()
    print("\n✓ Validation mAP curves displayed")
else:
    print("Warning: Could not display validation mAP curves")
    print("This may be because map_curve.png was not generated.")

In [ ]:
# ============================================================================
# Visualization 4: F1-Score Curve
# ============================================================================

print("\n=== F1-Score Curve ===")
print("=" * 80)

fig = evaluator.plot_f1_curve(str(output_dir))
if fig is not None:
    plt.show()
    print("\n✓ F1-Score curve displayed")
else:
    print("Warning: Could not generate F1-Score curve")

In [ ]:
# ============================================================================
# Visualization 5: Confusion Matrix
# ============================================================================

print("\n=== Confusion Matrix ===")
print("=" * 80)

fig = evaluator.plot_confusion_matrix(str(output_dir / 'evaluation'))
if fig is not None:
    plt.show()
    print("\n✓ Confusion matrix displayed")
else:
    print("Warning: Could not generate confusion matrix")

In [ ]:
# ============================================================================
# Analysis: Model Characteristics and Performance
# ============================================================================

print("\n" + "=" * 80)
print("MODEL ANALYSIS")
print("=" * 80)

# Model characteristics
total_params = sum(p.numel() for p in model.parameters())

print(f"\nModel Characteristics:")
print(f"  Backbone: ResNet50+FPN (Standard)")
print(f"  Input Size: {MODEL_CONFIG_V1['min_size']}x{MODEL_CONFIG_V1['max_size']}")
print(f"  Total Parameters: ~{total_params/1e6:.1f}M")
print(f"  Customization: None (Baseline)")
print(f"  Pretrained Weights: {MODEL_CONFIG_V1['pretrained']}")

print(f"\nPerformance Summary:")
print(f"  Best Training Loss: {history['best_loss']:.4f}")
print(f"  Total Epochs: {history['total_epochs']}")
print(f"  Test mAP@0.5: {metrics['mAP50']:.4f}")
print(f"  Test mAP@0.5:0.95: {metrics['mAP50_95']:.4f}")
print(f"  Test Precision: {metrics['precision']:.4f}")
print(f"  Test Recall: {metrics['recall']:.4f}")
f1 = 2 * metrics['precision'] * metrics['recall'] / (metrics['precision'] + metrics['recall'] + 1e-6)
print(f"  Test F1-Score: {f1:.4f}")

In [ ]:
# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED")
print("=" * 80)

print(f"\nExperiment: V1 - Faster R-CNN Baseline")
print(f"Model: Standard Faster R-CNN with ResNet50+FPN")
print(f"Pretrained: {MODEL_CONFIG_V1['pretrained']}")
print(f"Customization: None (Baseline)")

print(f"\n" + "-" * 80)
print(f"Training Performance:")
print(f"  Best Training Loss: {history['best_loss']:.4f}")
print(f"  Total Epochs: {history['total_epochs']}")

print(f"\n" + "-" * 80)
print(f"Test Set Performance:")
print(f"  mAP@0.5: {metrics['mAP50']:.4f}")
print(f"  mAP@0.5:0.95: {metrics['mAP50_95']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
f1_final = 2 * metrics['precision'] * metrics['recall'] / (metrics['precision'] + metrics['recall'] + 1e-6)
print(f"  F1-Score: {f1_final:.4f}")

print(f"\n" + "-" * 80)
print(f"Output Directory: {output_dir}")
print("\n" + "=" * 80)
print("✓ All experiments completed successfully")
print("=" * 80)